In [16]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

receipts = [
    ['라면', '계란', '우유'],
    ['라면', '계란'],
    ['라면', '계란', '콜라'],
    ['우유', '빵'],
    ['라면', '콜라'],
]

# ① 표로 바꾸기 — 샀으면 True ( 모든 아이템을 중복 없이 뽑아서 정렬된 리스트) 
items = sorted({i for r in receipts for i in r})
items

# {i for r in receipts for i in r}

# for (List<String> r : receipts) {   // 영수증 5장을 하나씩
#     for (String i : r) {            // 그 안의 아이템을 하나씩
#         result.add(i);
#     }
# }


['계란', '라면', '빵', '우유', '콜라']

In [5]:
df = pd.DataFrame([{i: (i in r) for i in items} for r in receipts])
df


# [{i: (i in r) for i in items} for r in receipts]
#
# for (List<String> r : receipts) {     // 영수증 5장 = 행 5개
#     Map<String,Boolean> row = new LinkedHashMap<>();
#     for (String i : items) {          // 아이템 5개 = 열 5개
#         row.put(i, r.contains(i));
#     }
#     table.add(row);
# }

,계란,라면,빵,우유,콜라
0,True,True,False,True,False
1,True,True,False,False,False
2,True,True,False,False,True
3,False,False,True,True,False
4,False,True,False,False,True


In [22]:
# ② 자주 같이 나오는 조합 찾기
# 최소지지도 : 전체 거래 중에서 그 조합이 등장한 비율(min_support=0.4) , 40%
freq = apriori(df, min_support=0.5, use_colnames=True)
freq

,support,itemsets
0,0.6,(계란)
1,0.8,(라면)
2,0.6,"(라면, 계란)"


In [23]:
# ③ 규칙 뽑기 — 향상도 1 이상만
# association_rules : 경우의 수 만들기(모든 조합경우수만듦 (ex (계란,라면) => 계란->라면, 라면->계란))
# metric='lift' : 향상도(관계가있는지)
# min_threshold : 최소 임계값(이 값 이상인 규칙) 1.0 : 향상도가 최소 1 이상인 조합만 꺼냄 
rules = association_rules(freq, metric='lift', min_threshold=1.0)

# 'antecedents':조건선행, 'consequents' 조건 후행, 'support' : 지지도, 'confidence' : 신뢰도 , 'lift' : 향상도
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

  antecedents consequents  support  confidence  lift
0        (라면)        (계란)      0.6        0.75  1.25
1        (계란)        (라면)      0.6        1.00  1.25


In [ ]:
# - 향상도 > 1 : 양의 관계 (같이 살 확률 높음)
# - 향상도 = 1 : 무관 (우연)
# - 향상도 < 1 : 음의 관계 (오히려 같이 안 삼)

In [24]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

data = [['라면','계란'], ['라면','계란','콜라'], ['라면','콜라'], ['우유','빵'], ['라면','계란']]
te = TransactionEncoder()
df = pd.DataFrame(te.fit(data).transform(data), columns=te.columns_)   # 0/1 표로

df

,계란,라면,빵,우유,콜라
0,True,True,False,False,False
1,True,True,False,False,True
2,False,True,False,False,True
3,False,False,True,True,False
4,True,True,False,False,False


In [25]:
freq = apriori(df, min_support=0.4, use_colnames=True)         # 자주 나온 조합
freq

,support,itemsets
0,0.6,(계란)
1,0.8,(라면)
2,0.4,(콜라)
3,0.6,"(라면, 계란)"
4,0.4,"(라면, 콜라)"


In [26]:
rules = association_rules(freq, metric='confidence', min_threshold=0.6)
print(rules[['antecedents','consequents','support','confidence','lift']])

  antecedents consequents  support  confidence  lift
0        (라면)        (계란)      0.6        0.75  1.25
1        (계란)        (라면)      0.6        1.00  1.25
2        (콜라)        (라면)      0.4        1.00  1.25


In [27]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

data = [
    ['우유','빵','버터'], ['빵','버터'], ['우유','빵'], ['우유','빵','버터','계란'],
    ['빵','계란'], ['우유','버터'], ['빵','버터','계란'], ['우유','빵','계란'],
]
te = TransactionEncoder()
df = pd.DataFrame(te.fit(data).transform(data), columns=te.columns_)

df

,계란,버터,빵,우유
0,False,True,True,True
1,False,True,True,False
2,False,False,True,True
3,True,True,True,True
4,True,False,True,False
5,False,True,False,True
6,True,True,True,False
7,True,False,True,True


In [30]:
freq = apriori(df, min_support=0.3, use_colnames=True)
freq

,support,itemsets
0,0.500,(계란)
1,0.625,(버터)
2,0.875,(빵)
3,0.625,(우유)
4,0.500,"(계란, 빵)"
5,0.500,"(버터, 빵)"
6,0.375,"(버터, 우유)"
7,0.500,"(우유, 빵)"


In [31]:
rules = association_rules(freq, metric='lift', min_threshold=1.0)
rules = rules.sort_values('lift', ascending=False)          # 향상도 높은 순
print(rules[['antecedents','consequents','support','confidence','lift']].head())

  antecedents consequents  support  confidence      lift
0        (계란)         (빵)      0.5    1.000000  1.142857
1         (빵)        (계란)      0.5    0.571429  1.142857


In [32]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 공개 미러 URL로 바로 불러오기 (Kaggle 'Groceries' - 마트 거래 9835건)
import urllib.request
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv'
lines = urllib.request.urlopen(url).read().decode('utf-8').strip().splitlines()
data = [ln.split(',') for ln in lines]         # 한 줄 = 한 번의 장바구니

te = TransactionEncoder()
df = pd.DataFrame(te.fit(data).transform(data), columns=te.columns_)
df


,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9830,False,False,False,False,False,False,False,False,False,True,...,False,False,False,True,False,False,False,True,False,False
9831,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9832,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
9833,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [36]:
freq = apriori(df, min_support=0.02, use_colnames=True)        # 2% 이상 등장
freq

,support,itemsets
0,0.033452,(UHT-milk)
1,0.052466,(beef)
2,0.033249,(berries)
3,0.026029,(beverages)
4,0.080529,(bottled beer)
...,...,...
117,0.032232,"(whipped/sour cream, whole milk)"
118,0.020742,"(whipped/sour cream, yogurt)"
119,0.056024,"(whole milk, yogurt)"
120,0.023183,"(root vegetables, whole milk, other vegetables)"


In [37]:
rules = association_rules(freq, metric='confidence', min_threshold=0.3)
rules = rules.sort_values('lift', ascending=False)
print(rules[['antecedents','consequents','support','confidence','lift']].head())

                       antecedents         consequents   support  confidence  \
34  (whole milk, other vegetables)   (root vegetables)  0.023183    0.309783   
32   (root vegetables, whole milk)  (other vegetables)  0.023183    0.474012   
17               (root vegetables)  (other vegetables)  0.047382    0.434701   
19            (whipped/sour cream)  (other vegetables)  0.028876    0.402837   
35            (whole milk, yogurt)  (other vegetables)  0.022267    0.397459   

        lift  
34  2.842082  
32  2.449770  
17  2.246605  
19  2.081924  
35  2.054131  
